In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config


In [0]:
%run ../00-common/03.silver_helpers

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.constructors"
silver_table=f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as F

In [0]:

constructors_df=(spark.table(bronze_table)).filter(F.col("batch_id")==v_batch_id)


In [0]:
constructors_dropped_df=constructors_df.drop('url')

In [0]:
constructors_renamed_df=(
    constructors_dropped_df
    .withColumnsRenamed({"constructorId":"constructor_id",
                         "name":"constructor_name",
                        })
)


In [0]:
constructors_distinct_df=constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
constructors_final_df=(constructors_distinct_df
    .withColumn('nationality',F.initcap(F.col("nationality"))))

In [0]:
write_to_silver(
    input_df=constructors_final_df,
    target_table=silver_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=["constructor_id","constructor_name","nationality", "ingestion_timestamp","source_file","batch_id"] 
)

In [0]:
# (
#     constructors_final_df
#     .write
#     .format("delta")
#     .mode("overwrite")
#     .saveAsTable(silver_table)
# )

In [0]:
display(spark.table(silver_table))